# 🏍 Motor Image Scraper — YOLO Dataset

Script ini mengunduh **50–100 gambar per kelas motor** dari DuckDuckGo & Bing Image Search  
untuk kebutuhan training model **YOLO**.

---
### Total kelas:
- **Yamaha**: YZF-R25, YZF-R3, MT-25, MT-03, Tenere 700, Tracer 9 GT, XMAX 250, TMAX 530, TMAX 560
- **Honda**: CBR250R–CBR1000RR-R, CB250R–Hornet 750, CRF250L–Africa Twin, X-ADV, Forza, Rebel
- **Kawasaki**: Ninja (250–H2 Carbon), Z-series, Versys, KLR, KLX, KX, Vulcan, W800
- **Suzuki**: Gixxer, GSX-R series, GSX-S series, V-Strom, Boulevard, Hayabusa
- **KTM**: Duke 250–990, RC 390, Adventure 250–890, Super Duke R
- **Husqvarna**: Svartpilen, Vitpilen, Norden 901
- **BMW Motorrad**: G310, S1000RR, M1000RR, F/R/K series
- **Benelli**: TNT, Leoncino, TRK, Imperiale, 752S
- **CFMOTO**: NK, MT, GT, SR, CL-X series
- **QJMotor**: Fort, SRV, SRK, Tourino
- **Moto Guzzi**: V7, V85 TT, V100 Mandello
- **Ducati**: Panigale, Monster, Streetfighter, Multistrada, DesertX, Scrambler, Diavel
- **Triumph**: Speed, Trident, Tiger, Street/Speed Triple, Bonneville, Scrambler, Rocket 3
- **Royal Enfield**: Classic, Hunter, Meteor, Bullet, Himalayan, Guerrilla, Interceptor, GT
- **Harley-Davidson**: Sportster, Nightster, Cruiser, Pan America, Touring
- **Indian Motorcycle**: Scout, Chief, Challenger, Pursuit, FTR
- **Aprilia**: RS, Tuono, Tuareg
- **Moto Morini**: X-Cape, Seiemmezzo, Calibro
- **MV Agusta**: Brutale, Dragster, F3, Turismo Veloce, Superveloce

## 📦 Install Dependencies

In [ ]:
# Install semua library yang dibutuhkan
!pip install ddgs requests pillow tqdm -q

## ⚙️ Konfigurasi

In [ ]:
import os
import time
import hashlib
import requests
import random
import re
import urllib.parse
from pathlib import Path
from PIL import Image
import io

# ─── KONFIGURASI ────────────────────────────────────────────
OUTPUT_DIR    = "dataset_motor"  # folder output dataset
IMAGES_TARGET = 75               # target gambar per kelas (50–100)
MIN_SIZE_KB   = 10               # abaikan gambar < 10 KB
IMAGE_SIZE    = (640, 640)       # resize ke ukuran YOLO
DELAY_SEC     = 1.5              # delay antar request
# ─────────────────────────────────────────────────────────────

Path(OUTPUT_DIR).mkdir(exist_ok=True)
print(f'✅ Output dir: {Path(OUTPUT_DIR).resolve()}')
print(f'🎯 Target   : {IMAGES_TARGET} gambar/kelas')

## 📋 Daftar Motor

In [ ]:
MOTOR_LIST = {
    "Yamaha": {
        "Sport": ["YZF-R25", "YZF-R3", "MT-25", "MT-03"],
        "Adventure_Touring": ["Tenere 700", "Tracer 9 GT"],
        "Maxi_Scooter": ["XMAX 250", "TMAX 530", "TMAX 560"],
    },
    "Honda": {
        "Sport": ["CBR250R", "CBR250RR", "CBR500R", "CBR650R",
                  "CBR600RR", "CBR1000RR Fireblade", "CBR1000RR-R Fireblade SP"],
        "Naked": ["CB250R", "CB300R", "CB500F", "CB650R", "CB1000R", "Hornet 750"],
        "Adventure": ["CRF250 Rally", "CRF300 Rally", "CRF250L",
                       "CRF300L", "CRF1100L Africa Twin"],
        "Adventure_Scooter": ["X-ADV 750"],
        "Maxi_Scooter": ["Forza 250", "Forza 350"],
        "Cruiser": ["Rebel 500", "Rebel 1100"],
    },
    "Kawasaki": {
        "Sport": ["Ninja 250", "Ninja 250SL", "Ninja ZX-25R", "Ninja ZX-4RR",
                  "Ninja ZX-4R", "Ninja ZX-6R", "Ninja ZX-10R",
                  "Ninja H2 SX", "Ninja H2", "Ninja H2 Carbon"],
        "Naked": ["Z250", "Z300", "Z400", "Z650", "Z650RS",
                  "Z900", "Z900RS", "Z1000", "Z H2"],
        "Adventure": ["Versys-X 250", "Versys 650", "Versys 1000", "KLR650"],
        "Dual_Purpose": ["KLX230R", "KLX250", "KLX300", "KX250", "KX450"],
        "Cruiser": ["Vulcan S 650"],
        "Retro": ["W800"],
    },
    "Suzuki": {
        "Sport": ["Gixxer 250", "Gixxer SF 250", "GSX250R",
                  "GSX-8R", "GSX-R600", "GSX-R750", "GSX-R1000"],
        "Naked": ["GSX-S250", "GSX-S750", "GSX-S1000", "GSX-8S"],
        "Adventure": ["V-Strom 250 SX", "V-Strom 650",
                       "V-Strom 800DE", "V-Strom 1050"],
        "Cruiser": ["Boulevard M109R"],
        "Hyperbike": ["Hayabusa 1300"],
    },
    "KTM": {
        "Naked": ["Duke 250", "Duke 390", "Duke 790", "Duke 890",
                  "Duke 990", "Super Duke R 1290"],
        "Sport": ["RC 390"],
        "Adventure": ["Adventure 250", "Adventure 390",
                       "Adventure 790", "Adventure 890"],
    },
    "Husqvarna": {
        "Naked": ["Svartpilen 250", "Svartpilen 401",
                  "Vitpilen 250", "Vitpilen 401"],
        "Adventure": ["Norden 901"],
    },
    "BMW_Motorrad": {
        "Sport": ["G310R", "G310GS", "S1000RR", "M1000RR"],
        "Adventure": ["F750GS", "F850GS", "F900GS", "R1250GS", "R1300GS"],
        "Touring": ["R1250RT", "K1600GT", "K1600GTL"],
        "Scooter": ["C400X", "C400GT"],
        "Roadster": ["F900R", "R1250R"],
    },
    "Benelli": {
        "Naked": ["TNT25", "TNT250", "Leoncino 250", "Leoncino 500",
                  "TRK251", "TRK502", "502C", "Imperiale 400", "752S"],
    },
    "CFMOTO": {
        "Naked": ["250 NK", "300 NK", "450 NK", "650 NK", "800 NK"],
        "Sport": ["450 SR"],
        "Adventure": ["450 MT", "650 MT", "800 MT"],
        "Retro": ["700 CL-X"],
        "Touring": ["650 GT"],
    },
    "QJMotor": {
        "Naked": ["Fort 250", "SRV250", "SRV250 Libero",
                  "SRV250 AMT", "SRV600V"],
        "Adventure": ["Fort 250 Adventure"],
        "Sport": ["SRK800RR"],
        "Touring": ["Tourino 700 SX"],
    },
    "Moto_Guzzi": {
        "Retro": ["V7 Stone"],
        "Adventure": ["V85 TT"],
        "Naked": ["V100 Mandello"],
    },
    "Ducati": {
        "Sport": ["Panigale V2", "Panigale V4", "SuperSport 950"],
        "Naked": ["Monster", "Streetfighter V2", "Streetfighter V4"],
        "Adventure": ["Multistrada V2", "Multistrada V4", "DesertX"],
        "Scrambler": ["Scrambler Icon", "Scrambler Full Throttle",
                       "Scrambler Nightshift"],
        "Cruiser": ["Diavel V4", "XDiavel"],
    },
    "Triumph": {
        "Naked": ["Speed 400", "Trident 660",
                  "Street Triple 765", "Speed Triple 1200"],
        "Adventure": ["Tiger Sport 660", "Tiger 900", "Tiger 1200"],
        "Scrambler": ["Scrambler 400 X", "Scrambler 900"],
        "Retro": ["Bonneville T100", "Bonneville T120"],
        "Cruiser": ["Rocket 3"],
    },
    "Royal_Enfield": {
        "Retro": ["Classic 350", "Hunter 350", "Meteor 350", "Bullet 350"],
        "Adventure": ["Himalayan 450", "Guerrilla 450"],
        "Cafe_Racer": ["Interceptor 650", "Continental GT 650",
                        "Super Meteor 650", "Shotgun 650"],
    },
    "Harley_Davidson": {
        "Sport": ["Sportster S", "Nightster"],
        "Cruiser": ["Street Bob", "Fat Bob", "Fat Boy", "Breakout",
                    "Heritage Classic", "Low Rider S", "Low Rider ST"],
        "Adventure": ["Pan America 1250"],
        "Touring": ["Road Glide", "Street Glide", "Road King"],
    },
    "Indian_Motorcycle": {
        "Cruiser": ["Scout", "Scout Bobber", "Chief",
                    "Springfield", "Chieftain"],
        "Touring": ["Challenger", "Pursuit", "Roadmaster"],
        "Naked": ["FTR"],
    },
    "Aprilia": {
        "Sport": ["RS457", "RS660", "RSV4"],
        "Naked": ["Tuono 457", "Tuono 660", "Tuono V4"],
        "Adventure": ["Tuareg 660"],
    },
    "Moto_Morini": {
        "Adventure": ["X-Cape 650"],
        "Retro": ["Seiemmezzo STR", "Seiemmezzo SCR", "Calibro"],
    },
    "MV_Agusta": {
        "Naked": ["Brutale 800", "Dragster 800"],
        "Sport": ["F3 800", "Superveloce"],
        "Touring": ["Turismo Veloce"],
    },
}

total_classes = sum(
    len(models)
    for cats in MOTOR_LIST.values()
    for models in cats.values()
)
print(f'📋 Total kelas motor : {total_classes}')
print(f'📦 Estimasi total gambar: ~{total_classes * IMAGES_TARGET}')

## 🔧 Fungsi Helper

In [ ]:
def make_class_name(brand: str, model: str) -> str:
    raw = f"{brand}_{model}"
    clean = raw.replace(" ", "_").replace("-", "_").replace("/", "_")
    clean = "".join(c for c in clean if c.isalnum() or c == "_")
    return clean.lower()

def image_hash(data: bytes) -> str:
    return hashlib.md5(data).hexdigest()

def is_valid_image(data: bytes) -> bool:
    if len(data) < MIN_SIZE_KB * 1024:
        return False
    try:
        img = Image.open(io.BytesIO(data))
        img.verify()
        return True
    except Exception:
        return False

def save_image(data: bytes, save_path: Path) -> bool:
    try:
        img = Image.open(io.BytesIO(data)).convert("RGB")
        img = img.resize(IMAGE_SIZE, Image.LANCZOS)
        img.save(str(save_path), "JPEG", quality=90)
        return True
    except Exception:
        return False

print('✅ Helper functions loaded')

## 🌐 Fungsi Scraping

In [ ]:
def scrape_duckduckgo(query: str, save_dir: Path,
                      target: int, existing_hashes: set) -> int:
    """Scraping dari DuckDuckGo Image Search."""
    try:
        from ddgs import DDGS
    except ImportError:
        print("  [!] Install: pip install ddgs")
        return 0

    saved = 0
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 Chrome/124.0.0.0 Safari/537.36"
        )
    }
    try:
        with DDGS() as ddgs:
            results = list(ddgs.images(
                query,
                max_results=target + 30,
            ))
    except Exception as e:
        print(f"  [!] DDG error: {e}")
        return 0

    for res in results:
        if saved >= target:
            break
        url = res.get("image", "")
        if not url:
            continue
        try:
            resp = requests.get(url, headers=headers, timeout=10)
            if resp.status_code != 200:
                continue
            data = resp.content
            if not is_valid_image(data):
                continue
            h = image_hash(data)
            if h in existing_hashes:
                continue
            existing_hashes.add(h)
            fname = save_dir / f"{h[:16]}.jpg"
            if save_image(data, fname):
                saved += 1
            time.sleep(random.uniform(0.3, 0.8))
        except Exception:
            continue

    return saved


def scrape_bing(query: str, save_dir: Path,
                target: int, existing_hashes: set) -> int:
    """Fallback scraping dari Bing Image Search."""
    saved = 0
    offset = 0
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 Chrome/124.0.0.0 Safari/537.36"
        )
    }
    while saved < target:
        enc_q = urllib.parse.quote(query)
        url = (
            f"https://www.bing.com/images/search"
            f"?q={enc_q}&first={offset}&count=30"
            f"&qft=+filterui:imagesize-large"
        )
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            if resp.status_code != 200:
                break
            img_urls = re.findall(r'"murl":"(https?://[^"]+)"', resp.text)
            if not img_urls:
                break
            for img_url in img_urls:
                if saved >= target:
                    break
                try:
                    r = requests.get(img_url, headers=headers, timeout=10)
                    if r.status_code != 200:
                        continue
                    data = r.content
                    if not is_valid_image(data):
                        continue
                    h = image_hash(data)
                    if h in existing_hashes:
                        continue
                    existing_hashes.add(h)
                    fname = save_dir / f"bing_{h[:16]}.jpg"
                    if save_image(data, fname):
                        saved += 1
                    time.sleep(random.uniform(0.3, 0.8))
                except Exception:
                    continue
            offset += 30
            time.sleep(DELAY_SEC)
        except Exception as e:
            print(f"  [!] Bing error: {e}")
            break
    return saved


def scrape_motor(brand: str, category: str, model: str, idx: int, total: int):
    """Scraping gambar untuk satu model motor."""
    class_name = make_class_name(brand, model)
    save_dir   = Path(OUTPUT_DIR) / class_name
    save_dir.mkdir(parents=True, exist_ok=True)

    existing = list(save_dir.glob("*.jpg"))
    existing_hashes: set = set()
    for f in existing:
        try:
            existing_hashes.add(image_hash(f.read_bytes()))
        except Exception:
            pass

    already_have = len(existing)
    if already_have >= IMAGES_TARGET:
        print(f"[{idx}/{total}] ✓ Skip — {class_name} ({already_have} gambar)")
        return already_have

    need = IMAGES_TARGET - already_have
    brand_clean = brand.replace("_", " ")
    queries = [
        f"{brand_clean} {model} motorcycle",
        f"{brand_clean} {model} motor",
        f"{brand_clean} {model} side view",
    ]
    query = random.choice(queries)
    print(f"[{idx}/{total}] ⟶ {brand_clean} {model}  | butuh {need} gambar...")

    count = already_have

    # Coba DuckDuckGo
    got = scrape_duckduckgo(query, save_dir, need, existing_hashes)
    count += got
    print(f"       DDG: +{got}  (total: {count})")

    # Fallback Bing jika masih kurang
    if count < IMAGES_TARGET:
        still_need = IMAGES_TARGET - count
        alt_query  = f"{brand_clean} {model} bike photo"
        got2 = scrape_bing(alt_query, save_dir, still_need, existing_hashes)
        count += got2
        print(f"       Bing: +{got2}  (total: {count})")

    status = "✅" if count >= IMAGES_TARGET * 0.8 else "⚠️"
    print(f"       {status} Selesai: {class_name} → {count} gambar")
    return count


print('✅ Scraping functions loaded')

## 🚀 Jalankan Scraping

> ⏱ Estimasi waktu: **~3–5 jam** untuk semua kelas (tergantung koneksi internet)
> 
> Script otomatis **resume** dari kelas yang sudah selesai jika dijalankan ulang.

In [ ]:
import hashlib

results = {}
idx     = 0

total_classes = sum(
    len(models)
    for cats in MOTOR_LIST.values()
    for models in cats.values()
)

print(f'🏍 Mulai scraping {total_classes} kelas motor...\n')
print('=' * 60)

for brand, categories in MOTOR_LIST.items():
    print(f'\n── Brand: {brand.replace("_", " ")} ──')
    for category, models in categories.items():
        for model in models:
            idx += 1
            class_name = make_class_name(brand, model)
            count = scrape_motor(brand, category, model, idx, total_classes)
            results[class_name] = count
            time.sleep(DELAY_SEC)

print('\n' + '=' * 60)
print('✅ SCRAPING SELESAI!')

## 📊 Generate classes.txt & Laporan

In [ ]:
# Simpan classes.txt untuk YOLO
classes = sorted(results.keys())
classes_path = Path(OUTPUT_DIR) / "classes.txt"
classes_path.write_text("\n".join(classes), encoding="utf-8")
print(f"📄 classes.txt → {len(classes)} kelas")

# Laporan ringkasan
print("\n" + "=" * 60)
print("LAPORAN SCRAPING")
print("=" * 60)

ok      = sum(1 for c in results.values() if c >= IMAGES_TARGET * 0.8)
fail    = [(k, v) for k, v in results.items() if v < IMAGES_TARGET * 0.8]
total_i = sum(results.values())

print(f"Total kelas   : {len(results)}")
print(f"Kelas OK (≥80%): {ok}")
print(f"Kelas kurang  : {len(fail)}")
print(f"Total gambar  : {total_i:,}")

if fail:
    print("\nKelas yang kurang gambar:")
    for name, cnt in sorted(fail, key=lambda x: x[1]):
        print(f"  ⚠️  {name}: {cnt} gambar")

# Simpan ke file
report_lines = ["Kelas,Jumlah Gambar,Status"]
for cls, cnt in sorted(results.items()):
    status = "OK" if cnt >= IMAGES_TARGET * 0.8 else "KURANG"
    report_lines.append(f"{cls},{cnt},{status}")

report_path = Path(OUTPUT_DIR) / "scraping_report.csv"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print(f"\n📊 Laporan CSV: {report_path}")

## 🔄 Retry Kelas yang Kurang Gambar

Jalankan cell ini jika ada kelas yang gambarnya kurang dari target.

In [ ]:
# Cek kelas mana yang masih kurang dan coba scrape ulang
retry_results = {}
idx = 0

for brand, categories in MOTOR_LIST.items():
    for category, models in categories.items():
        for model in models:
            class_name = make_class_name(brand, model)
            save_dir   = Path(OUTPUT_DIR) / class_name
            current    = len(list(save_dir.glob("*.jpg"))) if save_dir.exists() else 0

            if current < IMAGES_TARGET * 0.8:
                idx += 1
                print(f"🔄 Retry [{idx}]: {class_name} ({current} gambar)")
                count = scrape_motor(brand, category, model, idx, "?")
                retry_results[class_name] = count
                time.sleep(DELAY_SEC * 2)

if not retry_results:
    print("✅ Semua kelas sudah memenuhi target!")
else:
    print(f"\n🔄 Retry selesai. {len(retry_results)} kelas diproses ulang.")